# 1 Импорт библиотек и функций

## 1.1 Библиотеки

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from catboost import CatBoostClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import GridSearchCV
from tqdm.auto import tqdm

import warnings
warnings.filterwarnings('ignore')

## 1.2 Функции

In [4]:
def evaluate_classification(y_test, y_pred_proba):
    """Оценивает результаты классификации"""
    # Кодируем строковые метки в числа
    le = LabelEncoder()
    y_test_encoded = le.fit_transform(y_test)
    
    # Предсказанные классы
    y_pred = np.argmax(y_pred_proba, axis=1)
    
    # Метрики
    accuracy = accuracy_score(y_test_encoded, y_pred)
    f1 = f1_score(y_test_encoded, y_pred, average='weighted')
    
    # Для многоклассовой классификации
    auc_roc = roc_auc_score(y_test_encoded, y_pred_proba, multi_class='ovr', average='weighted')
    
    print(f"Accuracy: {accuracy:.4f}")
    print(f"F1-score: {f1:.4f}")
    print(f"AUC-ROC: {auc_roc:.4f}")
    
    print(f"\nClassification Report:")
    print(classification_report(y_test_encoded, y_pred, target_names=['L (падение)', 'N (нейтр)', 'R (рост)']))

In [7]:
def train_predict_rf(X_train, X_test, y_train):
    """Обучает и предсказывает Random Forest с подбором гиперпараметров"""
    le = LabelEncoder()
    y_train_encoded = le.fit_transform(y_train)
    
    # Упрощенная сетка для большого количества признаков
    param_grid = {
        'n_estimators': [100, 200, 350],
        'max_depth': [5, 8, 12],
        'min_samples_split': [10],
        'min_samples_leaf': [4]
    }
    
    print("Random Forest - подбор гиперпараметров...")
    model = RandomForestClassifier(random_state=13, n_jobs=-1)
    grid_search = GridSearchCV(model, param_grid, cv=3, scoring='f1_weighted', n_jobs=-1, verbose=1)
    grid_search.fit(X_train, y_train_encoded)
    
    print(f"Random Forest - лучшие параметры: {grid_search.best_params_}")
    print(f"Random Forest - лучшее качество: {grid_search.best_score_:.4f}")
    
    y_pred_proba = grid_search.predict_proba(X_test)
    return y_pred_proba

def train_predict_dt(X_train, X_test, y_train):
    """Обучает и предсказывает Decision Tree с подбором гиперпараметров"""
    le = LabelEncoder()
    y_train_encoded = le.fit_transform(y_train)
    
    # Упрощенная сетка с регуляризацией для многих признаков
    param_grid = {
        'max_depth': [10, 15, 20],
        'min_samples_split': [10, 20],
        'min_samples_leaf': [5, 10],
        'max_features': [0.3, 0.5, 'sqrt'],
        'criterion': ['gini']
    }
    
    print("Decision Tree - подбор гиперпараметров...")
    model = DecisionTreeClassifier(random_state=13)
    grid_search = GridSearchCV(model, param_grid, cv=3, scoring='f1_weighted', n_jobs=-1, verbose=1)
    grid_search.fit(X_train, y_train_encoded)
    
    print(f"Decision Tree - лучшие параметры: {grid_search.best_params_}")
    print(f"Decision Tree - лучшее качество: {grid_search.best_score_:.4f}")
    
    y_pred_proba = grid_search.predict_proba(X_test)
    return y_pred_proba

def train_predict_logreg(X_train, X_test, y_train):
    """Обучает и предсказывает Logistic Regression с регуляризацией"""
    le = LabelEncoder()
    y_train_encoded = le.fit_transform(y_train)
    
    # Сетка с акцентом на регуляризацию для многих признаков
    param_grid = {
        'C': [0.01, 0.1, 1, 10],
        'solver': ['liblinear', 'saga'],
        'penalty': ['l1', 'l2'],
        'max_iter': [1000]
    }
    
    print("Logistic Regression - подбор гиперпараметров...")
    model = LogisticRegression(random_state=13)
    grid_search = GridSearchCV(model, param_grid, cv=3, scoring='f1_weighted', n_jobs=-1, verbose=1)
    grid_search.fit(X_train, y_train_encoded)
    
    print(f"Logistic Regression - лучшие параметры: {grid_search.best_params_}")
    print(f"Logistic Regression - лучшее качество: {grid_search.best_score_:.4f}")
    
    # Выводим информацию о регуляризации
    best_model = grid_search.best_estimator_
    if hasattr(best_model, 'coef_'):
        non_zero_features = np.sum(best_model.coef_ != 0)
        print(f"Logistic Regression - ненулевых коэффициентов: {non_zero_features}/{X_train.shape[1]}")
    
    y_pred_proba = grid_search.predict_proba(X_test)
    return y_pred_proba

def train_predict_catboost(X_train, X_test, y_train):
    """Обучает и предсказывает CatBoost с подбором гиперпараметров"""
    le = LabelEncoder()
    y_train_encoded = le.fit_transform(y_train)
    
    # Упрощенная сетка для CatBoost
    param_grid = {
        'depth': [4, 6],
        'learning_rate': [0.03, 0.05, 0.1],
        'iterations': [300],
    }
    
    print("CatBoost - подбор гиперпараметров...")
    model = CatBoostClassifier(verbose=False, random_state=13, thread_count=-1)
    grid_search = GridSearchCV(model, param_grid, cv=3, scoring='f1_weighted', n_jobs=-1, verbose=1)
    grid_search.fit(X_train, y_train_encoded)
    
    print(f"CatBoost - лучшие параметры: {grid_search.best_params_}")
    print(f"CatBoost - лучшее качество: {grid_search.best_score_:.4f}")
    
    y_pred_proba = grid_search.predict_proba(X_test)
    return y_pred_proba

def train_predict_linear_svc(X_train, X_test, y_train):
    """Дополнительная функция: LinearSVC с L1-регуляризацией для отбора признаков"""
    le = LabelEncoder()
    y_train_encoded = le.fit_transform(y_train)
    
    param_grid = {
        'C': [0.01, 0.1, 1],
        'penalty': ['l1'],
        'loss': ['squared_hinge'],
        'dual': [False],
        'max_iter': [1000]
    }
    
    print("LinearSVC - подбор гиперпараметров...")
    from sklearn.svm import LinearSVC
    model = LinearSVC(random_state=13)
    grid_search = GridSearchCV(model, param_grid, cv=3, scoring='f1_weighted', n_jobs=-1, verbose=1)
    grid_search.fit(X_train, y_train_encoded)
    
    print(f"LinearSVC - лучшие параметры: {grid_search.best_params_}")
    print(f"LinearSVC - лучшее качество: {grid_search.best_score_:.4f}")
    
    # Для SVC нужно получить вероятности через calibration
    from sklearn.calibration import CalibratedClassifierCV
    calibrated_clf = CalibratedClassifierCV(grid_search.best_estimator_, cv=3)
    calibrated_clf.fit(X_train, y_train_encoded)
    
    y_pred_proba = calibrated_clf.predict_proba(X_test)
    return y_pred_proba

In [9]:
def train_test_split_by_date(df, target_column, test_size=0.2):
    """
    Разбивает данные на train/test по дате и возвращает X, y
    
    Args:
        df: DataFrame с колонкой 'begin'
        target_column: название целевой переменной
        test_size: доля тестовых данных (0.2 = 20%)
    """
    df = df.sort_values('begin').reset_index(drop=True)
    
    # Вычисляем индекс разбиения
    split_idx = int(len(df) * (1 - test_size))
    
    train_df = df.iloc[:split_idx].copy()
    test_df = df.iloc[split_idx:].copy()
    
    print(f"Train: {train_df['begin'].min()} - {train_df['begin'].max()} ({len(train_df)} samples)")
    print(f"Test:  {test_df['begin'].min()} - {test_df['begin'].max()} ({len(test_df)} samples)")
    
    # Удаляем колонку 'begin' и разделяем на X, y
    X_train = train_df.drop(columns=['begin', target_column])
    y_train = train_df[target_column]
    
    X_test = test_df.drop(columns=['begin', target_column])
    y_test = test_df[target_column]
    
    print(f"Признаков: {X_train.shape[1]}")
    
    return X_train, X_test, y_train, y_test

# 2 Подготовка данных

## 2.0 Список тикеров

In [13]:
tickers = [
    'SBER', 'GAZP'
]

## 2.1 Чтение

### 2.1.1 Тех признаки

In [17]:
data_SBER = pd.read_csv("../../../data/stock_features_data/stocks_features_SBER.csv")
data_GAZP = pd.read_csv("../../../data/stock_features_data/stocks_features_GAZP.csv")

### 2.1.2 Новости

In [20]:
news = pd.read_csv("../../../data/news/3_titles_fasttext.csv")
news.head(2)

,begin,heading_interfax_fasttext_0,heading_interfax_fasttext_1,heading_interfax_fasttext_2,heading_interfax_fasttext_3,heading_interfax_fasttext_4,heading_interfax_fasttext_5,heading_interfax_fasttext_6,heading_interfax_fasttext_7,heading_interfax_fasttext_8,...,heading_kommersant_fasttext_290,heading_kommersant_fasttext_291,heading_kommersant_fasttext_292,heading_kommersant_fasttext_293,heading_kommersant_fasttext_294,heading_kommersant_fasttext_295,heading_kommersant_fasttext_296,heading_kommersant_fasttext_297,heading_kommersant_fasttext_298,heading_kommersant_fasttext_299
0,2022-05-01 10:00:00,0.124890,-0.101783,-0.017166,0.030840,0.006648,-0.104627,-0.015941,0.008354,0.027082,...,-0.046253,0.002917,0.0167,-0.066125,-0.008185,0.015039,0.09441,0.007845,-0.005977,-0.035276
1,2022-05-01 12:00:00,0.079931,-0.120844,0.019048,-0.042986,0.004966,-0.124700,-0.002416,-0.023473,0.068674,...,0.000000,0.000000,0.0000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000


## 2.2 Удаление лишних признаков

In [23]:
# Для SBER
part_SBER = data_SBER[['begin', 'close', 'target_price_change']]
data_SBER.drop(['close', 'target_price_change'], axis=1, inplace=True)

# Для GAZP
part_GAZP = data_GAZP[['begin', 'close', 'target_price_change']]
data_GAZP.drop(['close', 'target_price_change'], axis=1, inplace=True)

## 2.3 Объединение признаков

In [26]:
data_SBER = data_SBER.merge(news, on='begin', how='left')
data_GAZP = data_GAZP.merge(news, on='begin', how='left')

## 2.4 Разделение на train/test

In [29]:
X_train_SBER, X_test_SBER, y_train_SBER, y_test_SBER = train_test_split_by_date(
    df=data_SBER, 
    target_column='target_class',
    test_size=0.2
)

X_train_GAZP, X_test_GAZP, y_train_GAZP, y_test_GAZP = train_test_split_by_date(
    df=data_GAZP, 
    target_column='target_class',
    test_size=0.2
)

Train: 2022-05-31 18:00:00 - 2025-03-11 14:00:00 (3427 samples)
Test:  2025-03-11 16:00:00 - 2025-09-30 18:00:00 (857 samples)
Признаков: 910
Train: 2022-05-31 18:00:00 - 2025-03-11 14:00:00 (3427 samples)
Test:  2025-03-11 16:00:00 - 2025-09-30 18:00:00 (857 samples)
Признаков: 910


# 3 Обучение моделей с подбором гиперпараметров

## 3.1 Дерево решений

### 3.1.1 Сбер

In [36]:
y_pred_SBER = train_predict_dt(X_train_SBER, X_test_SBER, y_train_SBER)
evaluate_classification(y_test_SBER, y_pred_SBER)

Decision Tree - подбор гиперпараметров...
Fitting 3 folds for each of 36 candidates, totalling 108 fits
Decision Tree - лучшие параметры: {'criterion': 'gini', 'max_depth': 15, 'max_features': 'sqrt', 'min_samples_leaf': 10, 'min_samples_split': 10}
Decision Tree - лучшее качество: 0.3554
Accuracy: 0.3186
F1-score: 0.3209
AUC-ROC: 0.4886

Classification Report:
              precision    recall  f1-score   support

 L (падение)       0.28      0.39      0.33       269
   N (нейтр)       0.44      0.31      0.37       379
    R (рост)       0.24      0.23      0.24       209

    accuracy                           0.32       857
   macro avg       0.32      0.31      0.31       857
weighted avg       0.34      0.32      0.32       857



### 3.1.2 Газпром

In [38]:
y_pred_GAZP = train_predict_dt(X_train_GAZP, X_test_GAZP, y_train_GAZP)
evaluate_classification(y_test_GAZP, y_pred_GAZP)

Decision Tree - подбор гиперпараметров...
Fitting 3 folds for each of 36 candidates, totalling 108 fits
Decision Tree - лучшие параметры: {'criterion': 'gini', 'max_depth': 10, 'max_features': 0.5, 'min_samples_leaf': 10, 'min_samples_split': 10}
Decision Tree - лучшее качество: 0.3637
Accuracy: 0.2964
F1-score: 0.2943
AUC-ROC: 0.4433

Classification Report:
              precision    recall  f1-score   support

 L (падение)       0.27      0.46      0.34       285
   N (нейтр)       0.44      0.23      0.31       464
    R (рост)       0.10      0.12      0.11       108

    accuracy                           0.30       857
   macro avg       0.27      0.27      0.25       857
weighted avg       0.34      0.30      0.29       857



## Вывод

## 3.2 Регрессия

### 3.2.1 Сбер

In [43]:
y_pred_SBER = train_predict_logreg(X_train_SBER, X_test_SBER, y_train_SBER)
evaluate_classification(y_test_SBER, y_pred_SBER)

Logistic Regression - подбор гиперпараметров...
Fitting 3 folds for each of 16 candidates, totalling 48 fits
Logistic Regression - лучшие параметры: {'C': 10, 'max_iter': 1000, 'penalty': 'l1', 'solver': 'liblinear'}
Logistic Regression - лучшее качество: 0.3172
Logistic Regression - ненулевых коэффициентов: 2040/910
Accuracy: 0.3442
F1-score: 0.3462
AUC-ROC: 0.5093

Classification Report:
              precision    recall  f1-score   support

 L (падение)       0.32      0.33      0.33       269
   N (нейтр)       0.43      0.32      0.37       379
    R (рост)       0.28      0.41      0.33       209

    accuracy                           0.34       857
   macro avg       0.35      0.35      0.34       857
weighted avg       0.36      0.34      0.35       857



### 3.2.2 Газпром

In [45]:
y_pred_GAZP = train_predict_logreg(X_train_GAZP, X_test_GAZP, y_train_GAZP)
evaluate_classification(y_test_GAZP, y_pred_GAZP)

Logistic Regression - подбор гиперпараметров...
Fitting 3 folds for each of 16 candidates, totalling 48 fits
Logistic Regression - лучшие параметры: {'C': 10, 'max_iter': 1000, 'penalty': 'l1', 'solver': 'liblinear'}
Logistic Regression - лучшее качество: 0.3109
Logistic Regression - ненулевых коэффициентов: 2083/910
Accuracy: 0.3956
F1-score: 0.3966
AUC-ROC: 0.5154

Classification Report:
              precision    recall  f1-score   support

 L (падение)       0.36      0.55      0.43       285
   N (нейтр)       0.55      0.37      0.44       464
    R (рост)       0.11      0.11      0.11       108

    accuracy                           0.40       857
   macro avg       0.34      0.34      0.33       857
weighted avg       0.43      0.40      0.40       857



## Вывод

## 3.3 Вот он, лес

### 3.3.1 Сбер

In [32]:
y_pred_SBER = train_predict_rf(X_train_SBER, X_test_SBER, y_train_SBER)
evaluate_classification(y_test_SBER, y_pred_SBER)

Random Forest - подбор гиперпараметров...
Fitting 3 folds for each of 9 candidates, totalling 27 fits
Random Forest - лучшие параметры: {'max_depth': 8, 'min_samples_leaf': 4, 'min_samples_split': 10, 'n_estimators': 100}
Random Forest - лучшее качество: 0.2955
Accuracy: 0.3349
F1-score: 0.2499
AUC-ROC: 0.4852

Classification Report:
              precision    recall  f1-score   support

 L (падение)       0.32      0.85      0.46       269
   N (нейтр)       0.46      0.15      0.22       379
    R (рост)       0.23      0.01      0.03       209

    accuracy                           0.33       857
   macro avg       0.34      0.34      0.24       857
weighted avg       0.36      0.33      0.25       857



### 3.3.2 Газпром

In [34]:
y_pred_GAZP = train_predict_rf(X_train_GAZP, X_test_GAZP, y_train_GAZP)
evaluate_classification(y_test_GAZP, y_pred_GAZP)

Random Forest - подбор гиперпараметров...
Fitting 3 folds for each of 9 candidates, totalling 27 fits
Random Forest - лучшие параметры: {'max_depth': 12, 'min_samples_leaf': 4, 'min_samples_split': 10, 'n_estimators': 100}
Random Forest - лучшее качество: 0.3115
Accuracy: 0.4434
F1-score: 0.4160
AUC-ROC: 0.4918

Classification Report:
              precision    recall  f1-score   support

 L (падение)       0.30      0.30      0.30       285
   N (нейтр)       0.53      0.63      0.57       464
    R (рост)       0.19      0.03      0.05       108

    accuracy                           0.44       857
   macro avg       0.34      0.32      0.31       857
weighted avg       0.41      0.44      0.42       857



## Вывод

## 3.4 Бустинг

### 3.4.1 Сбер

In [37]:
y_pred_SBER = train_predict_catboost(X_train_SBER, X_test_SBER, y_train_SBER)
evaluate_classification(y_test_SBER, y_pred_SBER)

CatBoost - подбор гиперпараметров...
Fitting 3 folds for each of 6 candidates, totalling 18 fits
CatBoost - лучшие параметры: {'depth': 4, 'iterations': 300, 'learning_rate': 0.03}
CatBoost - лучшее качество: 0.2644
Accuracy: 0.3816
F1-score: 0.3648
AUC-ROC: 0.5366

Classification Report:
              precision    recall  f1-score   support

 L (падение)       0.34      0.60      0.43       269
   N (нейтр)       0.46      0.35      0.40       379
    R (рост)       0.36      0.16      0.22       209

    accuracy                           0.38       857
   macro avg       0.39      0.37      0.35       857
weighted avg       0.40      0.38      0.36       857



### 3.4.2 Газпром

In [ ]:
y_pred_GAZP = train_predict_catboost(X_train_GAZP, X_test_GAZP, y_train_GAZP)
evaluate_classification(y_test_GAZP, y_pred_GAZP)

## Вывод